# Working with Planetary Computer data using obstore

This notebook walks through reading Planetary Computer data with [obstore](https://developmentseed.org/obstore/) — a Python library that talks to cloud object stores (Azure Blob, S3, GCS) directly, without going through HTTP wrappers like fsspec. It's the foundation that higher-level libraries (async-geotiff, zarr-python, deck.gl-raster via Lonboard) sit on top of.

**Five reasons to use it over the older `planetary_computer.sign() + fsspec` pattern:**

1. **Reliability** — SAS tokens auto-refresh. No `TokenExpiredError` mid-job, no manual re-signing.
2. **Cost** — range reads download only the bytes you need (e.g. a 16 KB COG header instead of a 100 MB file).
3. **Speed** — async surface fires reads in parallel. Roughly N× faster than serial for multi-file workloads.
4. **Composability** — the same store works with async-geotiff, zarr-python, Lonboard, etc.
5. **Portability** — `AzureStore`, `S3Store`, `GCSStore` are interchangeable. Cloud-agnostic code.

Each cell below calls out which of these it demonstrates. Speed-relevant cells use `%%time` so you can compare wall-clock numbers.

The companion [obstore tutorial](../overview/obstore.md) has the full narrative and migration reference.

## Install

obstore is the main library. `pystac-client` lets us query Planetary Computer's STAC API to find a scene to read. `requests` powers the sync credential provider; `aiohttp` + `aiohttp_retry` power the async one (we use both in this notebook).

In [ ]:
%pip install --quiet obstore pystac-client requests aiohttp aiohttp_retry

## Authenticate from a STAC asset

**Demonstrates: reliability.** `PlanetaryComputerCredentialProvider` handles SAS token acquisition and refresh under the hood — no manual `planetary_computer.sign()` calls anywhere in this notebook. If a token expires mid-job, the provider re-acquires it transparently. The old fsspec pattern required you to handle re-signing and retry logic yourself.

**Expected result:** working `provider` object, no output printed.

In [ ]:
import pystac_client
from obstore.auth.planetary_computer import PlanetaryComputerCredentialProvider

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)
item = next(catalog.search(collections=["naip"], max_items=1).items())
asset = item.assets["image"]

provider = PlanetaryComputerCredentialProvider.from_asset(asset)

Notice the asset href is unsigned — no SAS query string appended. The provider signs it for you at read time.

In [ ]:
asset.href

## Build a store

A *store* is obstore's connection to a specific cloud location. Once built, you hand it to any obstore read/write function — or to any higher-level library that accepts an obstore-compatible store.

**Expected result:** working `store` object, no output printed.

In [ ]:
from obstore.store import AzureStore

store = AzureStore(credential_provider=provider)

## Read

`from_asset()` scopes the store to that *specific blob* — the asset URL becomes the store's prefix. So every read uses an empty string as the path; obstore appends the path to the prefix, and you don't want it appending anything. (For multi-object access you'd build a container-scoped store instead — covered further down.)

Three ways to read, each demonstrating a different value prop:

### 1. Read the entire file

**Demonstrates: baseline (the slow path you want to avoid).** Use when you actually want all the bytes. NAIP scenes range 100–500 MB.

A surprise here: this cell is slow even on a fast connection. Azure Blob caps single-stream downloads at roughly 8–15 MB/s — your home bandwidth doesn't help. To go faster you'd need parallel range reads against the same file (which is what async-geotiff does internally when it reads COG tiles).

This cell is the foil for everything below. The whole point of range reads and async is to avoid this scenario.

**Expected result:** 100–500 million bytes, 30–90 seconds depending on which NAIP scene came back.

In [ ]:
%%time
import obstore

buf = obstore.get(store, "").bytes()
print(f"downloaded {len(buf):,} bytes")

### 2. Read a byte range (16 KB)

**Demonstrates: cost savings.** A Cloud Optimized GeoTIFF stores its header in the first few KB. Most libraries (async-geotiff, GDAL, rasterio) only need the header to start working — they don't need the pixel data until you ask for a specific window. Range reads make this possible.

**Expected result:** 16,384 bytes, well under a second. Tens of thousands of times less data than the full file above (the exact multiple depends on your scene's size).

In [ ]:
%%time
header = obstore.get_range(store, "", start=0, end=16384)
print(f"downloaded {len(header):,} bytes")
print(f"that's {len(buf) / len(header):,.0f}x less data than the full file")

### 3. Read multiple byte ranges in one request

**Demonstrates: latency savings.** When you need several slices of the same file — say, multiple COG tiles — you could issue separate `get_range` calls. Each one is a round-trip to Azure. `get_ranges` batches them into a single HTTP request, cutting round-trip latency.

**Expected result:** two ranges of 16 KB each, similar wall time to a single `get_range`.

In [ ]:
%%time
ranges = obstore.get_ranges(
    store, "", starts=[0, 65536], ends=[16384, 81920]
)
print([len(r) for r in ranges])

## Listing requires a container-scoped store

**Demonstrates: reach beyond a single asset.** Up to here we've worked with one blob. To enumerate objects under a prefix ("show me every NAIP scene in Montana in 2023"), the store needs to be scoped to the container *and* the credential provider needs container-level `List` permission. The asset-derived provider above only signs the single blob — it can't list — so we build a fresh provider against the container URL.

**Expected result:** three lines printed, each a blob path and its size in bytes.

In [ ]:
container_provider = PlanetaryComputerCredentialProvider(
    "https://naipeuwest.blob.core.windows.net/naip/"
)
container_store = AzureStore(
    account_name="naipeuwest",
    container_name="naip",
    credential_provider=container_provider,
)

for batch in obstore.list(container_store, prefix="v002/mt/2023/"):
    for entry in batch[:3]:
        print(entry["path"], entry["size"])
    break

## Concurrent reads (async) — the speed payoff

**Demonstrates: speed via parallelism.** Up to here, reads happen one at a time. For multi-file workloads, running them in parallel is dramatically faster than serial. Below we read the same 4 KB header four times — first serially, then concurrently — and compare wall times.

Async needs its own credential provider class (`PlanetaryComputerAsyncCredentialProvider`) backed by `aiohttp` instead of `requests`.

In [ ]:
import asyncio
from obstore.auth.planetary_computer import PlanetaryComputerAsyncCredentialProvider

async_provider = PlanetaryComputerAsyncCredentialProvider.from_asset(asset)
async_store = AzureStore(credential_provider=async_provider)

We'll time both with `time.perf_counter()` for an apples-to-apples comparison (the `%%time` magic doesn't play well with top-level `await`, so we measure manually).

**Warmup the async store.** First call has to acquire a SAS token from Planetary Computer — a separate HTTP round-trip. We do one throwaway read so that overhead doesn't pollute the timing below. (The sync store was already warmed up by cells 1/2/3, which is why we only need to warm the async store.)

In [ ]:
_ = await obstore.get_range_async(async_store, "", start=0, end=4096)
print("warmed up")

**Serial baseline:** eight reads, one after the other.

In [ ]:
import time

n_reads = 8
start = time.perf_counter()
for _ in range(n_reads):
    obstore.get_range(store, "", start=0, end=4096)
serial_elapsed = time.perf_counter() - start
print(f"serial ({n_reads} reads): {serial_elapsed:.3f}s")

**Concurrent:** same eight reads, all firing at once via `asyncio.gather`.

**Expected result:** several times faster than the serial cell above. The exact speedup depends on Azure's per-connection throttling, but you should see a clear win.

In [ ]:
async def fetch_header():
    return await obstore.get_range_async(async_store, "", start=0, end=4096)

start = time.perf_counter()
headers = await asyncio.gather(*[fetch_header() for _ in range(n_reads)])
concurrent_elapsed = time.perf_counter() - start

print(f"concurrent ({n_reads} reads): {concurrent_elapsed:.3f}s")
print(f"speedup: {serial_elapsed / concurrent_elapsed:.1f}x")
print(f"all {len(headers)} reads returned {len(headers[0])} bytes each")

The speedup scales with how many parallel reads you're doing. For real workloads (building a mosaic, fetching all bands across all scenes in an AOI), this is the difference between "minutes" and "seconds."

## Hand the store to async-geotiff

**Demonstrates: composability.** The whole point of obstore is that *other libraries* sit on top of it. Once you have a working `AzureStore`, you can hand it to any library that accepts an [obspec](https://github.com/developmentseed/obspec)-compatible store — async-geotiff, zarr-python, and others — and they'll read through your authenticated connection. No re-auth, no double signing.

In [ ]:
%pip install --quiet async-geotiff

Open the NAIP scene as a COG and read its metadata. `geotiff.transform` is the affine that maps pixel coordinates to geographic coordinates. `geotiff.crs` is the coordinate reference system.

**Expected result:** the affine transform and the CRS name (e.g. `NAD83 / UTM zone 11N` — varies by scene).

In [ ]:
from async_geotiff import GeoTIFF

# async_store is scoped to the asset, so path is "" (same rule as obstore reads above)
geotiff = await GeoTIFF.open("", store=async_store)
print(geotiff.transform)
print(geotiff.crs.name)

If you want the full CRS details (datum, axis order, area of use), just evaluate `geotiff.crs` on its own — pyproj prints a detailed dump.

Notice async-geotiff only fetched ~16 KB to get this metadata, not the full file. The range-read win compounds at every level of the stack.

## Bonus: portability

**Demonstrates: cloud-agnostic code.** The same `obstore.get(store, ...)` call works against S3 or GCS — only the store constructor changes. This cell isn't runnable (we don't have S3 creds), but it shows the shape:

```python
from obstore.store import S3Store

s3_store = S3Store(bucket="my-bucket", region="us-west-2")
buf = obstore.get(s3_store, "path/to/object").bytes()  # same call
```

Any library that accepts an obspec-compatible store benefits automatically. async-geotiff opening a COG works identically against Azure, S3, or GCS.

## You're done

If you got expected output on every cell above, the obstore stack is wired up end-to-end:

- **Reliability** — authenticated against Planetary Computer with auto-refreshing SAS tokens
- **Cost** — read 16 KB of a multi-hundred-MB file via range reads (tens of thousands of times less egress)
- **Latency** — batched multi-range read in a single round-trip
- **Speed** — parallel reads several times faster than serial
- **Composability** — handed the same store to async-geotiff, opened a COG with no re-auth
- **Portability** — same API shape works for S3 and GCS

From here, any obspec-compatible library plugs in the same way. Check the companion [Lonboard tutorial](../overview/lonboard.md) for interactive visualization or the [async-geotiff tutorial](../overview/async-geotiff.md) for pixel-level analysis.